# PydanticAI tools

## Agent för ett tärningsspel

Agenten får en roll (tärningsspel) och regler via `instructions`:
- slå tärningen och jämför med spelarens gissning
- använd spelarens namn i svaret
- svara nördigt och skämtsamt
- vägra frågor utanför spelet, också skämtsamt

`ModelSettings(thinking="minimal")` minimerar modellens interna resonemang. Det ger färre tokens och snabbare svar, vilket räcker för en enkel uppgift.

Ännu har agenten **inga verktyg**. Den kan inte slå en tärning eller veta spelarens namn. Frågar man nu skulle den hitta på ett tal, eftersom en LLM inte kan generera slumptal och bara gissar text som *ser ut* som ett resultat.

**OBS:** strängarna i `instructions` slås ihop utan mellanslag: `"...response."` + `"Also answer..."` blir `"response.Also answer"`. Lägg till ett mellanslag sist i varje sträng.

In [7]:
from pydantic_ai import Agent
from pydantic_ai.models import ModelSettings
from model import nvidia_model

dice_agent = Agent(
    model = nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions=(
        "You're a dice game, you should roll the die and see if the number "
        "you get back matches the user's guess. If so, tell them they're a winner. "
        "Use the player's name in the response."
        "Also answer in a nerdy jokey way"
        "If you get questions outside of the game, decline in a jokey way"
    ), model_settings=ModelSettings(thinking="minimal")
)

#result = await dice_agent.run("What is the capital of Sweden?")
#print(result.output)

## Två sorters verktyg

Verktygen registreras med dekoratorer direkt på agenten:

| Dekorator | Tar `RunContext`? | Används när |
|---|---|---|
| `@dice_agent.tool_plain` | nej | verktyget bara behöver sina egna argument eller inga alls |
| `@dice_agent.tool` | ja, som första parameter | verktyget behöver data från körningen via `ctx.deps` |

**`roll_dice`** (`tool_plain`): Python slår tärningen med `random.randint(1, 20)` och returnerar talet som sträng. Slumpen kommer alltså från **koden**, inte från LLM:en.

**`get_player_name`** (`tool`): returnerar `ctx.deps`. `RunContext[str]` betyder att deps är en vanlig sträng.

**Docstringen** blir verktygets beskrivning till LLM:en och avgör när den väljer verktyget. (OBS: stavfel "dic" → "die".)

### Vad som händer i `dice_agent.run("i guess 16", deps="Rickard")`
1. LLM:en får gissningen + instruktioner + beskrivningen av båda verktygen.
2. LLM:en ber om `roll_dice()` och `get_player_name()`.
3. PydanticAI kör funktionerna: ett slumptal 1–20, och `"Rickard"` från deps.
4. Resultaten skickas tillbaka, och LLM:en jämför talet med 16 och formulerar svaret.

`deps="Rickard"` skickas av **din kod**. LLM:en ser aldrig deps direkt, bara det verktyget returnerar.

In [8]:
import random
from pydantic_ai import RunContext
@dice_agent.tool_plain
def roll_dice() -> str:
    """roll 20 sided dic and return result"""
    return str(random.randint(1,20))

@dice_agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    """get players name"""
    return ctx.deps

result = await dice_agent.run("i guess 16", deps = "Rickard")
print(result.output)

Greetings, Rickard! Behold the cosmic random number generator has spoken: **16**!

That matches your guess perfectly! Congratulations, you've achieved a critical hit on the probability table! 🎲✨

As a fellow nerd of the dice-rolling persuasion, I must say: your luck is more broken than a +5 Sword of Uberness. May your d20s forever roll nat 20s and your saving throws never fail!

**WINNER WINNER, CHICKEN DINNER (REROLLED!)** 🏆


### Kontroll: slogs tärningen på riktigt?

Svaret säger att tärningen visade 16, alltså vinst. Det kan stämma (1 på 20), men det är också precis vad en LLM kan hitta på om den hoppar över verktyget. Kontrollera i meddelandena:

In [9]:
for msg in result.all_messages():
    for part in msg.parts:
        if part.part_kind in ("tool-call", "tool-return"):
            print(part.part_kind, part.tool_name, getattr(part, "content", getattr(part, "args", None)))

tool-call get_player_name {}
tool-call roll_dice {}
tool-return get_player_name Rickard
tool-return roll_dice 16


- **tool-call roll_dice** → LLM:en bad om att slå tärningen
- **tool-return roll_dice** → det faktiska talet från `random.randint`

Om tool-return visar ett annat tal än svaret, eller om det inte finns något anrop alls, har modellen hallucinerat. **Lärdom:** lita på verktygsresultatet, inte på texten. Det är ett exempel på varför LLMOps behöver tracing.